# Feature-Extraction Candidate Screening

**Issue #11 (M1-EDA):** before M2 builds the feature-extraction pipeline, check whether the
"obvious next" time-domain statistics — skewness, crest factor, peak-to-peak — actually add
information beyond the RMS and kurtosis already computed in #9/#10, using this dataset's own
data rather than a generic vibration-analysis checklist. Frequency-domain candidates are
reasoned about but not computed here (see Section 5) — no FFT/spectral analysis has been done
on this dataset yet, so nothing frequency-domain is asserted to work.

Builds on:
- **#9** (`01_vibration_signal_evolution.ipynb`): RMS/kurtosis per file, cached in
  `data/processed/`. `1st_test`'s inner-race failure is impulsive (peak kurtosis 74.6) but only
  moderately visible in RMS amplitude (peak ratio 2.87x); `2nd_test`/`3rd_test`'s outer-race
  failures are the reverse (RMS peak ratio 6.3x/7.2x, kurtosis 17.1/19.7).
- **#10** (`02_health_state_labeling.ipynb`, merged): the Normal/Degrading/Critical labeling
  rule — `1.3x` baseline RMS for onset, a per-experiment geometric-midpoint multiplier for the
  Critical boundary. Reused here (recomputed from the same cached RMS, not hardcoded) so this
  screening can be broken down by health state.

## 1. Setup

Same per-file granularity as #9/#10 (one value per snapshot file, for the one documented-failure
channel per experiment) — the new statistics are computed at the same resolution the existing
labels live at, so they're directly comparable.

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, skew

DATA_DIR = Path("../data/raw")
CACHE_DIR = Path("../data/processed")

ONSET_MULTIPLE = 1.3
ROLLING_WINDOW = 10
BASELINE_N_FILES = 50

EXPERIMENTS = {
    "1st_test": {"dir": DATA_DIR / "1st_test", "channel_idx": 4,
                 "bearing_label": "Bearing 3, Ch 5", "failure_mode": "inner race defect",
                 "color": "#0072B2"},
    "2nd_test": {"dir": DATA_DIR / "2nd_test", "channel_idx": 0,
                 "bearing_label": "Bearing 1, Ch 1", "failure_mode": "outer race failure",
                 "color": "#D55E00"},
    "3rd_test": {"dir": DATA_DIR / "3rd_test", "channel_idx": 2,
                 "bearing_label": "Bearing 3, Ch 3", "failure_mode": "outer race failure",
                 "color": "#009E73"},
}

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def list_snapshot_files(test_dir: Path) -> list[Path]:
    return sorted(test_dir.iterdir(), key=lambda p: p.name)


def parse_timestamp(path: Path) -> datetime:
    return datetime.strptime(path.name, "%Y.%m.%d.%H.%M.%S")


def load_channel(path: Path, channel_idx: int) -> np.ndarray:
    return (
        pd.read_csv(path, sep="\t", header=None, usecols=[channel_idx], dtype=np.float32)
        .iloc[:, 0]
        .to_numpy()
    )

## 2. Compute the candidate statistics

Skewness, crest factor (peak / RMS), and peak-to-peak are computed per file alongside RMS and
kurtosis, for the same three tracked channels as #9/#10. Cached separately from #9's
`*_rms_kurtosis.csv` (a different, already-closed issue's artifact) to
`data/processed/*_candidate_features.csv`.

In [2]:
def compute_candidate_features(name: str, cfg: dict, force: bool = False) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"{name}_candidate_features.csv"
    if cache_path.exists() and not force:
        return pd.read_csv(cache_path, parse_dates=["timestamp"])

    files = list_snapshot_files(cfg["dir"])
    records = []
    for i, f in enumerate(files):
        sig = load_channel(f, cfg["channel_idx"])
        rms = float(np.sqrt(np.mean(sig**2)))
        peak = float(np.abs(sig).max())
        records.append({
            "timestamp": parse_timestamp(f),
            "file_index": i,
            "rms": rms,
            "kurtosis": float(kurtosis(sig, fisher=False)),
            "skewness": float(skew(sig)),
            "crest_factor": peak / rms if rms > 0 else float("nan"),
            "peak_to_peak": float(sig.max() - sig.min()),
        })
        if (i + 1) % 1000 == 0:
            print(f"  {name}: {i + 1}/{len(files)} files processed")

    df = pd.DataFrame(records)
    df.to_csv(cache_path, index=False)
    return df


stats = {name: compute_candidate_features(name, cfg) for name, cfg in EXPERIMENTS.items()}
for name, df in stats.items():
    print(f"{name}: {len(df)} files")

1st_test: 2156 files
2nd_test: 984 files
3rd_test: 6324 files


In [3]:
# Reproduce #10's labels from the cached RMS alone (not hardcoded) so this notebook stays
# self-contained and the Critical multiplier is re-derived, not trusted from memory.
CRITICAL_SPAN_FRACTION = 0.5

for name, df in stats.items():
    base_rms = df["rms"].head(BASELINE_N_FILES).mean()
    rolling_rms = df["rms"].rolling(ROLLING_WINDOW, min_periods=1).mean()
    ratio = rolling_rms / base_rms
    peak_ratio = ratio.max()
    critical_multiple = ONSET_MULTIPLE * (peak_ratio / ONSET_MULTIPLE) ** CRITICAL_SPAN_FRACTION

    label = np.where(
        ratio > critical_multiple, "Critical",
        np.where(ratio > ONSET_MULTIPLE, "Degrading", "Normal"),
    )
    df["rms_ratio"] = ratio
    df["label"] = pd.Categorical(label, categories=["Normal", "Degrading", "Critical"], ordered=True)
    df.attrs["critical_multiple"] = critical_multiple
    df.attrs["baseline_rms"] = base_rms

    print(f"{name}: critical_multiple={critical_multiple:.3f}  "
          f"counts={df['label'].value_counts().reindex(['Normal','Degrading','Critical']).to_dict()}")

1st_test: critical_multiple=1.931  counts={'Normal': 1979, 'Degrading': 160, 'Critical': 17}
2nd_test: critical_multiple=2.867  counts={'Normal': 657, 'Degrading': 304, 'Critical': 23}
3rd_test: critical_multiple=3.049  counts={'Normal': 6158, 'Degrading': 99, 'Critical': 67}


Multipliers and label counts match #10 exactly (`1.93x / 2.87x / 3.05x`, `1979/160/17`,
`657/304/23`, `6158/99/67`) — confirms this notebook's independent recomputation agrees with the
merged labeling notebook.

## 3. Does peak-to-peak add anything beyond RMS?

Peak-to-peak is the amplitude range within a file. The natural worry: for a roughly symmetric
signal this is just RMS rescaled by a near-constant factor, in which case it's redundant.

In [4]:
for name, df in stats.items():
    corr_full = df["peak_to_peak"].corr(df["rms"])
    deg_crit = df[df["label"] != "Normal"]
    corr_deg = deg_crit["peak_to_peak"].corr(deg_crit["rms"])
    pp_ratio = df["peak_to_peak"] / df["rms"]
    print(f"{name}: corr(peak_to_peak, rms) whole-life={corr_full:.2f}, "
          f"Degrading+Critical only={corr_deg:.2f}  |  "
          f"peak_to_peak/rms: mean={pp_ratio.mean():.2f} cv={pp_ratio.std()/pp_ratio.mean():.3f}")

1st_test: corr(peak_to_peak, rms) whole-life=0.68, Degrading+Critical only=0.65  |  peak_to_peak/rms: mean=9.72 cv=0.472
2nd_test: corr(peak_to_peak, rms) whole-life=0.95, Degrading+Critical only=0.95  |  peak_to_peak/rms: mean=9.83 cv=0.101
3rd_test: corr(peak_to_peak, rms) whole-life=0.94, Degrading+Critical only=0.93  |  peak_to_peak/rms: mean=8.90 cv=0.094


**Checked and not recommended.** `peak_to_peak` correlates `0.65-0.95` with `rms` across all
three experiments, whole-life and within the Degrading+Critical window alike. For `2nd_test` and
`3rd_test` it is close to a constant multiple of RMS (coefficient of variation `0.09-0.10` — the
ratio barely moves). `1st_test` is somewhat less redundant (`cv=0.47`) but still correlates at
`0.65`. This is the weakest of the three candidates and adds negligible information RMS doesn't
already carry — excluded from the M2 shortlist.

## 4. Crest factor and skewness: correlation with kurtosis, in the region that matters

Whole-lifetime correlation is dominated by hundreds or thousands of flat, low-variance Normal
files, which can make a feature look more independent than it actually is where a classifier
would use it. The more honest check is correlation **restricted to Degrading+Critical files
only** — the region a health-state classifier actually has to discriminate within.

In [5]:
for name, df in stats.items():
    deg_crit = df[df["label"] != "Normal"]
    corr = deg_crit[["rms", "kurtosis", "skewness", "crest_factor"]].corr()
    print(f"=== {name}  (n={len(deg_crit)} Degrading+Critical files) ===")
    print(corr.round(2).to_string())
    print()

=== 1st_test  (n=177 Degrading+Critical files) ===
               rms  kurtosis  skewness  crest_factor
rms           1.00     -0.10     -0.02         -0.14
kurtosis     -0.10      1.00     -0.42          0.88
skewness     -0.02     -0.42      1.00         -0.41
crest_factor -0.14      0.88     -0.41          1.00

=== 2nd_test  (n=327 Degrading+Critical files) ===
               rms  kurtosis  skewness  crest_factor
rms           1.00      0.65     -0.49          0.39
kurtosis      0.65      1.00     -0.49          0.57
skewness     -0.49     -0.49      1.00         -0.39
crest_factor  0.39      0.57     -0.39          1.00

=== 3rd_test  (n=166 Degrading+Critical files) ===
               rms  kurtosis  skewness  crest_factor
rms           1.00      0.63     -0.25          0.58
kurtosis      0.63      1.00     -0.74          0.78
skewness     -0.25     -0.74      1.00         -0.59
crest_factor  0.58      0.78     -0.59          1.00



Two things stand out.

**`rms` vs `kurtosis` decouple sharply for `1st_test` specifically** (`corr = -0.10`, essentially
independent) but stay correlated for `2nd_test`/`3rd_test` (`0.65`, `0.63`). This is the concrete
version of #9's observation that `1st_test`'s inner-race failure is impulsive rather than
amplitude-driven: inside its own degradation window, how loud the signal gets and how spiky it
gets are nearly unrelated. That is the evidence for keeping kurtosis as a feature **alongside**
RMS, not a generic "kurtosis is a standard vibration feature" assertion — for two of the three
experiments it would be substantially redundant with RMS.

**`crest_factor` vs `kurtosis` correlate `0.57-0.88`** across all three experiments in this
window — consistently substantial. Crest factor is mostly restating what kurtosis already
captures, in the range that matters for the classifier.

In [6]:
print("crest_factor by label (mean [min .. max]):\n")
for name, df in stats.items():
    print(f"{name}:")
    for lab in ["Normal", "Degrading", "Critical"]:
        sub = df.loc[df["label"] == lab, "crest_factor"]
        if len(sub):
            print(f"   {lab:10s} n={len(sub):5d}  mean={sub.mean():5.2f}  [{sub.min():.2f} .. {sub.max():.2f}]")
    print()

crest_factor by label (mean [min .. max]):

1st_test:
   Normal     n= 1979  mean= 5.31  [3.41 .. 23.95]
   Degrading  n=  160  mean=11.77  [4.87 .. 25.95]
   Critical   n=   17  mean= 9.67  [6.05 .. 15.69]

2nd_test:
   Normal     n=  657  mean= 5.19  [4.10 .. 7.72]
   Degrading  n=  304  mean= 5.09  [3.82 .. 7.49]
   Critical   n=   23  mean= 5.80  [2.38 .. 9.33]

3rd_test:
   Normal     n= 6158  mean= 4.69  [3.72 .. 10.61]
   Degrading  n=   99  mean= 4.83  [3.77 .. 6.09]
   Critical   n=   67  mean= 5.58  [1.77 .. 10.09]



`1st_test` shows something worth flagging explicitly: mean crest factor **rises** from Normal
(5.31) to Degrading (11.77) but then **falls** in Critical (9.67) — non-monotonic with severity.
This matches a known crest-factor pattern for progressing bearing faults (it's most sensitive to
early, isolated impacts; once the defect produces near-continuous impact energy, RMS catches up
faster than peak amplitude does, so the ratio comes back down) — but it means crest factor alone
cannot be read as "higher = worse," which matters if it were ever hand-thresholded the way RMS
was in #10.

**Verdict: crest factor is a low-priority candidate.** It is inexpensive to compute, but the
evidence here shows it mostly duplicates kurtosis where it counts (`0.57-0.88` correlation in the
Degrading+Critical window across all three experiments) and behaves non-monotonically in the one
experiment where it might have added the most. Worth including in M2's feature-importance /
redundancy check rather than assumed useful, and not worth hand-designing a threshold around.

## 5. Skewness: a genuinely different signal, with a caveat on how to threshold it

In [7]:
print("baseline (first 50 files) stats, all near-Gaussian / near-symmetric as expected:\n")
for name, df in stats.items():
    base_kurt = df["kurtosis"].head(BASELINE_N_FILES).mean()
    base_skew_abs = df["skewness"].head(BASELINE_N_FILES).abs().mean()
    print(f"  {name}: baseline kurtosis={base_kurt:.2f}  baseline |skewness|={base_skew_abs:.4f}")

print("\nskewness by label (mean [min .. max]):\n")
for name, df in stats.items():
    print(f"{name}:")
    for lab in ["Normal", "Degrading", "Critical"]:
        sub = df.loc[df["label"] == lab, "skewness"]
        if len(sub):
            print(f"   {lab:10s} n={len(sub):5d}  mean={sub.mean():6.3f}  [{sub.min():.3f} .. {sub.max():.3f}]")
    print()

baseline (first 50 files) stats, all near-Gaussian / near-symmetric as expected:

  1st_test: baseline kurtosis=3.37  baseline |skewness|=0.0261
  2nd_test: baseline kurtosis=3.48  baseline |skewness|=0.0291
  3rd_test: baseline kurtosis=3.47  baseline |skewness|=0.0330

skewness by label (mean [min .. max]):

1st_test:
   Normal     n= 1979  mean=-0.008  [-1.718 .. 1.065]
   Degrading  n=  160  mean=-0.077  [-2.542 .. 1.233]
   Critical   n=   17  mean=-0.139  [-0.315 .. 0.067]

2nd_test:
   Normal     n=  657  mean= 0.001  [-0.114 .. 0.085]
   Degrading  n=  304  mean=-0.101  [-0.520 .. 0.121]
   Critical   n=   23  mean=-0.130  [-0.766 .. 0.580]

3rd_test:
   Normal     n= 6158  mean= 0.014  [-0.133 .. 0.114]
   Degrading  n=   99  mean=-0.155  [-0.659 .. 0.167]
   Critical   n=   67  mean=-0.169  [-1.648 .. 0.137]



`2nd_test` and `3rd_test` show a real, ordered trend: mean skewness goes from ~0 (Normal) to
increasingly negative through Degrading and Critical. It is not just restating kurtosis — within
the Degrading+Critical window (Section 4's table), `corr(skewness, kurtosis)` is `-0.42`
(`1st_test`), `-0.49` (`2nd_test`), `-0.74` (`3rd_test`): a real relationship, but far from
collinear, especially for the first two.

**The caveat:** all three experiments have baseline `|skewness|` around `0.03` — essentially
zero, unlike RMS/kurtosis which have strictly positive, well-scaled baselines. #9/#10's whole
approach was thresholds expressed as **a multiple of baseline**. That pattern breaks for
skewness: "3x a number near zero" is a tiny absolute value, trivially crossed by ordinary noise,
so it is not a meaningful onset signal by itself. M2 needs an absolute threshold (or a smoothed/
windowed version) for skewness, not a naive reuse of the ratio-to-baseline pattern.

In [8]:
# A concrete illustration of the pre-onset transient in 1st_test, worth a closer look:
name = "1st_test"
df = stats[name]
onset_idx = int(df.index[df["label"] != "Normal"][0])
pre_onset = df.iloc[:onset_idx]
spikes = pre_onset[pre_onset["skewness"].abs() > 1.0]
print(f"{name}: onset at file {onset_idx}. Pre-onset files with |skewness| > 1.0 "
      f"(vs. a baseline mean of ~0.03): {len(spikes)}")
print(spikes[["file_index", "skewness", "kurtosis", "rms"]].to_string(index=False))

1st_test: onset at file 1906. Pre-onset files with |skewness| > 1.0 (vs. a baseline mean of ~0.03): 5
 file_index  skewness  kurtosis      rms
       1845 -1.251029 32.637825 0.168604
       1846 -1.490284 40.581371 0.167163
       1863 -1.717811 59.526493 0.174424
       1886 -1.255649 34.882961 0.169065
       1905 -1.364849 45.282944 0.175390


Five files, all in the last ~60 files before `1st_test`'s official onset (file 1906) — and all
with visibly elevated kurtosis (33-60) too. This lines up with kurtosis's own early crossing
(Section 6): both higher-order statistics show transient spikes *before* the RMS-based rule
fires, in the one experiment where the failure mode is impulsive rather than amplitude-driven.

**Verdict: skewness is a plausible, worth-testing candidate**, not confirmed. The trend is real
and not redundant with kurtosis, but per-file skewness is noisy (compare the spread within each
label above) and its natural threshold scale is different from RMS/kurtosis's. M2 should test it
smoothed (e.g., the same 10-file rolling mean used for RMS) rather than raw per-file, and expect
to set its threshold in absolute terms.

## 6. Kurtosis as a leading vs. lagging indicator — it depends on the failure mode

#9 already showed kurtosis is a **corroborating** signal. Here's whether it's also an **earlier**
one than the RMS-based onset #10 labels on.

In [9]:
for name, df in stats.items():
    base_kurt = df["kurtosis"].head(BASELINE_N_FILES).mean()
    rms_onset_idx = int(df.index[df["label"] != "Normal"][0])

    kurt_candidates = df.index[df["kurtosis"] > 2 * base_kurt]
    kurt_onset_idx = int(kurt_candidates[0]) if len(kurt_candidates) else None

    if kurt_onset_idx is not None:
        gap_files = kurt_onset_idx - rms_onset_idx
        gap_hours = (
            df["timestamp"].iloc[rms_onset_idx] - df["timestamp"].iloc[kurt_onset_idx]
        ).total_seconds() / 3600
        lead_lag = "LEADS" if gap_files < 0 else "LAGS"
        print(f"{name}: RMS onset file {rms_onset_idx}, kurtosis > 2x baseline first at file "
              f"{kurt_onset_idx} -> kurtosis {lead_lag} by {abs(gap_files)} files "
              f"({abs(gap_hours):.1f}h)")
    else:
        print(f"{name}: kurtosis never exceeds 2x its baseline")

1st_test: RMS onset file 1906, kurtosis > 2x baseline first at file 1820 -> kurtosis LEADS by 86 files (17.0h)
2nd_test: RMS onset file 651, kurtosis > 2x baseline first at file 971 -> kurtosis LAGS by 320 files (53.3h)
3rd_test: RMS onset file 6158, kurtosis > 2x baseline first at file 6303 -> kurtosis LAGS by 145 files (24.2h)


Kurtosis **leads** RMS-based onset for `1st_test` (by ~17h) — consistent with the impulsive
failure showing up in signal shape before it shows up in amplitude — but **lags** for `2nd_test`
and `3rd_test` (by ~53h and ~24h) — for an amplitude-driven outer-race failure, RMS is the
earlier signal and kurtosis only becomes noticeably elevated once the fault is already well
underway.

This is a genuinely useful, failure-mode-dependent finding, and a caution against a generic
"kurtosis is an early-warning feature" claim: here, it is early for one failure mode and late for
the other two.

## 7. Frequency-domain features — flagged for M2, not evaluated here

No FFT or spectral analysis has been performed anywhere in this EDA. Everything above is a
time-domain statistic per 1-second snapshot.

There is a reasoned case for investigating frequency-domain features in M2, grounded in the
failure-mode labels already established (not asserted as something already shown to work):

- `1st_test`'s bearing failure is documented as an **inner race defect**; `2nd_test` and
  `3rd_test` are documented **outer race failures**. Classical bearing-fault theory predicts
  different characteristic defect frequencies for inner-race vs. outer-race faults (BPFI vs.
  BPFO), which time-domain RMS/kurtosis cannot distinguish between — they'd only show that
  *something* is wrong, not *what*.
- `1st_test`'s failure being impulsive-but-not-amplitude-heavy (Section 4) is exactly the
  signature spectral kurtosis / envelope analysis is designed to isolate (periodic impacts
  buried in broadband noise) — a plausible next step, not a confirmed one.

**M2 should investigate this, not assume a specific feature.** The dataset ships with the
sampling rate and shaft speed needed to compute the theoretical defect frequencies (`data/README.md`
/ the NASA readme PDF), so grounding any spectral feature in the actual documented fault, rather
than a generic FFT-peak-energy feature, is the right next step — but it requires new analysis
this notebook does not do.

## 8. Shortlist for M2

| Feature | Status | Evidence (this notebook / #9 / #10) |
|---|---|---|
| **RMS** | Confirmed useful | Already the basis of the #10 labeling rule; degradation trend visible and generalizable as a ratio-to-baseline across all three experiments. |
| **Kurtosis** | Confirmed useful | Decouples from RMS specifically in the impulsive failure (`1st_test`: `corr=-0.10` in the Degrading+Critical window, vs. `0.63-0.65` for the other two) — catches a failure mode RMS amplitude undersells. Failure-mode-dependent lead/lag vs. RMS onset (leads by ~17h for `1st_test`, lags by ~24-53h for `2nd_test`/`3rd_test`) — useful, but not uniformly an "early warning" feature. |
| **Skewness** | Plausible, worth testing | Real, non-redundant trend (moderate anti-correlation with kurtosis, not collinear) toward more-negative values with severity in `2nd_test`/`3rd_test`; shows pre-onset transient spikes in `1st_test` (5 files, elevated kurtosis too, all within ~60 files of onset). Needs smoothing and an absolute (not baseline-relative) threshold — its own baseline sits near zero across all three experiments, so the ratio-to-baseline pattern from #9/#10 does not transfer. |
| **Crest factor** | Plausible, low priority | Correlates `0.57-0.88` with kurtosis in the Degrading+Critical window across all three experiments — largely redundant where it counts. Non-monotonic with severity in `1st_test` (rises then falls). Cheap to compute; worth a feature-importance check in M2, not worth a hand-designed threshold. |
| **Peak-to-peak** | Checked, not recommended | Correlates `0.65-0.95` with RMS, whole-life and within Degrading+Critical alike; near-constant multiple of RMS for `2nd_test`/`3rd_test` (cv `0.09-0.10`). Adds negligible information beyond RMS. |
| **Frequency-domain (spectral kurtosis, BPFI/BPFO-aligned energy)** | Untested, flagged for M2 | No spectral analysis has been done. Plausible given the documented inner-race-vs-outer-race split, but nothing here confirms a specific feature works — M2 should investigate, not assume. |

Scope note: this notebook screens candidates: it doesn't implement the M2 extraction pipeline
(no `src/features/` code), and doesn't change anything about the labeling rule fixed in #10.